# 15 — Целевая модель ≤ 30B (Qwen-Coder vs облако)

> **Требование заказчика:** «модели до 30 миллиардов параметров,
> заказчик разворачивает в своём контуре с ограниченными ресурсами».
>
> **Решение (ADR-0008):** Qwen-Coder 32B primary, fallback на gpt-4o-mini,
> OpenAI-совместимый контракт LLMClient — переключение моделей одной
> строкой конфига.

## Что покажем

1. `LLMClient` абстракция — один контракт для всех моделей.
2. Mock-моделей: small (быстро, посредственно), medium, large (медленно, лучше).
3. Тот же task через 3 модели — видим trade-off.
4. Cost-калькулятор: цена прогона eval-set через каждую.
5. **Где LLM реально нужен** — кратко по проекту (5 точек).


## 🧒 Аналогия для ребёнка

У тебя в гараже три инструмента:
- **Маленькая отвёртка** — лёгкая, всегда с собой, но шуруп
  из бетонной стены не вытащит.
- **Средний шуруповёрт** — справляется с большинством, неудобно
  таскать каждый день.
- **Огромный перфоратор** — пробивает любую стену, но дома хранить
  негде, нужен прокат за 1000 ₽/день.

В LLM то же: маленькая (7B) для простых, средняя (32B) для
большинства, огромная (>100B) — когда нужна гарантия. Контракт
`LLMClient` — это **универсальный держатель**, в который можно
вставить любую: меняешь насадку, не меняешь руку.


## 1. LLMClient контракт


In [ ]:
"""
@brief Подготовка окружения и mock-БД через in-memory SQLite.
@details
    Никаких внешних зависимостей кроме stdlib + sqlite3 (есть в Colab из коробки).
    SQLite используем как «упрощённую модель PostgreSQL» — он умеет
    почти весь стандартный SQL, что достаточно для демонстраций уязвимостей.
@note
    Реальная система работает на PostgreSQL (см. ADR-0001),
    использует pglast для AST-парсинга. Здесь, для наглядности,
    эмулируем аудитор через `re` (регулярки) и простой pattern matching.
"""
import sqlite3
import re
import time
from textwrap import dedent


def section(title):
    """@brief Печатает заголовок секции."""
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def show_result(rows, max_rows=10):
    """@brief Печатает результаты запроса в виде таблицы."""
    if not rows:
        print("  (нет строк)")
        return
    for i, r in enumerate(rows[:max_rows]):
        print(f"  {i + 1:>3}. {r}")
    if len(rows) > max_rows:
        print(f"  ... ещё {len(rows) - max_rows} строк")


def print_finding(f):
    """@brief Красиво печатает Finding от нашего аудитора."""
    print(f"  ⚠️  {f['rule_id']}")
    print(f"      vuln_class:  {f['vuln_class']}")
    print(f"      severity:    {f['severity']}")
    print(f"      risk_score:  {f['risk_score']}/10")
    print(f"      message:     {f['message']}")
    if f.get("evidence_refs"):
        print(f"      ссылки:      {', '.join(f['evidence_refs'])}")


from dataclasses import dataclass
from typing import Optional


@dataclass
class LLMConfig:
    """@brief Универсальная конфигурация для любого OpenAI-compat провайдера."""
    model: str
    base_url: str = "https://api.openai.com/v1"
    api_key: str = "mock"
    temperature: float = 0.2
    max_tokens: int = 2048
    timeout_s: float = 30.0
    # «характеристики» (для mock)
    quality: float = 0.7        # 0..1
    latency_per_1k_tokens: float = 1.0  # «сек»/1k токенов
    cost_per_1m_tokens: float = 1.0     # $/1M


class LLMClient:
    """@brief Mock-клиент. В проде — httpx.post к /v1/chat/completions."""

    def __init__(self, cfg: LLMConfig):
        self.cfg = cfg

    ##
    # @brief Mock chat-completion.
    # @return  dict с полями text, tokens_in, tokens_out, latency_seconds, cost_usd.
    def chat(self, messages, sql_to_generate=None):
        # Имитируем — генерируем SQL разного качества
        tokens_in = sum(len(m.get("content", "")) for m in messages) // 4
        tokens_out = 200 if not sql_to_generate else len(sql_to_generate) // 4
        latency = (tokens_in + tokens_out) / 1000 * self.cfg.latency_per_1k_tokens
        cost = (tokens_in + tokens_out) / 1_000_000 * self.cfg.cost_per_1m_tokens
        # Mock-вывод
        if self.cfg.quality > 0.85:
            sql = "SELECT id, full_name FROM clients WHERE balance > 0 LIMIT 100"
        elif self.cfg.quality > 0.6:
            sql = "SELECT * FROM clients WHERE balance > 0 LIMIT 100"  # с одним багом (SELECT *)
        else:
            sql = "SELECT * FROM clients"  # совсем плохо
        return {
            "text": sql,
            "tokens_in": tokens_in,
            "tokens_out": tokens_out,
            "latency_seconds": latency,
            "cost_usd": cost,
        }


# Три «модели» — mock-конфиги
SMALL = LLMConfig(model="qwen2.5-7b-instruct",
                  quality=0.55,
                  latency_per_1k_tokens=0.3,
                  cost_per_1m_tokens=0.18)
MEDIUM = LLMConfig(model="qwen2.5-coder-32b-instruct",
                   quality=0.78,
                   latency_per_1k_tokens=0.8,
                   cost_per_1m_tokens=0.66)
LARGE = LLMConfig(model="gpt-4o-mini",
                  quality=0.88,
                  latency_per_1k_tokens=0.5,
                  cost_per_1m_tokens=0.75)


## 2. Один task через 3 модели


In [ ]:
task_messages = [
    {"role": "system", "content": "Ты — генератор PostgreSQL по NL-вопросам аналитиков. Возвращай только SQL."},
    {"role": "user",   "content": "покажи клиентов с положительным балансом, топ-100 по убыванию"},
]


def run_through(cfg, label):
    client = LLMClient(cfg)
    resp = client.chat(task_messages)
    print(f"  [{label:18s}]")
    print(f"    model:   {cfg.model}")
    print(f"    SQL:     {resp['text']}")
    print(f"    quality: ~{cfg.quality*100:.0f}%")
    print(f"    latency: {resp['latency_seconds']*1000:.0f} ms")
    print(f"    cost:    ${resp['cost_usd']:.6f}")
    print()


section("Один task через 3 модели")
run_through(SMALL,  "small (7B)")
run_through(MEDIUM, "medium (32B Qwen)")
run_through(LARGE,  "large (gpt-4o-mini)")


## 3. Cost-калькулятор на полный eval-set


In [ ]:
def estimate_eval_cost(cfg, n_questions=120, avg_in_tokens=15000, avg_out_tokens=400, iters_per_q=2.0):
    """@brief Оценка стоимости прогона eval-set ADR-0006."""
    per_q_tokens = (avg_in_tokens + avg_out_tokens) * iters_per_q
    total_tokens = per_q_tokens * n_questions
    cost = total_tokens / 1_000_000 * cfg.cost_per_1m_tokens
    latency_total = total_tokens / 1000 * cfg.latency_per_1k_tokens
    return cost, latency_total


section("Полный прогон eval-set (120 вопросов × 2 итерации)")
print(f"{'модель':<28} {'cost $':>10} {'wall time (мин)':>18}")
print("-" * 60)
for cfg, label in [(SMALL, "small (7B)"),
                   (MEDIUM, "medium (32B Qwen)"),
                   (LARGE, "large (gpt-4o-mini)")]:
    c, t = estimate_eval_cost(cfg)
    print(f"{cfg.model:<28} {c:>10.2f} {t/60:>18.1f}")


## 4. Переключение моделей одной строкой


In [ ]:
section("Переключение через DI — один и тот же код работает с любой моделью")

# В реальном проекте generator-узел LangGraph получает LLMClient через DI
class GeneratorNode:
    def __init__(self, client: LLMClient):
        self.client = client

    def __call__(self, task):
        return self.client.chat([
            {"role": "system", "content": "PG SQL generator"},
            {"role": "user",   "content": task},
        ])


for cfg, label in [(SMALL, "dev"), (MEDIUM, "prod-Qwen"), (LARGE, "prod-cloud")]:
    node = GeneratorNode(LLMClient(cfg))
    resp = node("ну там клиентов")
    print(f"  env={label:12s}  model={cfg.model:30s}  SQL: {resp['text']}")


## 5. Где LLM реально нужен (по проекту)

Ноутбуки 01-09 показали Phase 1 (детерминированный аудитор). LLM
нужен в **6 точках**:

| Узел                   | Кто работает               | Почему не алгоритм                    |
|------------------------|----------------------------|---------------------------------------|
| **generator**          | ✅ LLM (Qwen-32B)          | NL→SQL по схеме — задача синтеза      |
| **schema_link**        | ✅ Эмбеддинг (e5)          | Семантическая близость, не grep       |
| **few-shot retrieval** | ✅ Эмбеддинг               | то же                                 |
| **auditor Phase 2**    | ✅ LLM-судья (Qwen-32B)    | FP-фильтр + объяснение на русском     |
| **reflector**          | ✅ LLM (Qwen-7B)           | Парафраз findings → урок              |
| **dataset synthesis**  | ✅ LLM (одноразово, GPT-4o-mini) | SQL→NL back-translation               |

Phase 1 (правила) — это **скелет**. LLM — **мускулы**.


## Итог

Мы увидели проблему **под микроскопом** и **симуляцию решения** из ADR.

## Куда дальше

- **Описание проблемы:** [problems/engineering/06-on-prem-model-size/README.md](../problems/engineering/06-on-prem-model-size/README.md)
- **Варианты решения + почему так:** [problems/engineering/06-on-prem-model-size/solutions.md](../problems/engineering/06-on-prem-model-size/solutions.md)
- **Архитектура цикла:** [docs/adr/0002-loop-architecture-langgraph.md](../docs/adr/0002-loop-architecture-langgraph.md)
